# Ilaria RVC on Kaggle

Enable **GPU** and **Internet**, then click **Run All** in the editor. Open **Ilaria RVC** in the output of step 4.

Anyone with the link can use the app. Use a session without private data.

### 1. Check GPU

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

WORK = Path("/kaggle/working")
ROOT = WORK / "ilaria-rvc"
TOOLS = WORK / ".ilaria-notebook"
PYTHON = TOOLS / "venv/bin/python"
SOURCE_URL = "https://github.com/TheStingerX/Ilaria-RVC.git"
RELEASE = "v2.1.0"
COMMIT = "d68b8465a23c27d9af24bea4d188a19a7af51d0a"

def run(*args, **kwargs):
    return subprocess.run([str(arg) for arg in args], check=True, **kwargs)

if not WORK.is_dir() or sys.platform != "linux":
    raise RuntimeError("Run this notebook on Kaggle.")
if not shutil.which("nvidia-smi"):
    raise RuntimeError("Enable an NVIDIA GPU and restart the session.")
run("nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv")

TOOLS.mkdir(exist_ok=True)
if not ROOT.exists():
    print("Downloading Ilaria RVC. Enable Internet in Kaggle settings.", flush=True)
    run("git", "clone", "--depth", "1", "--branch", RELEASE, SOURCE_URL, ROOT)
if not (ROOT / ".git").is_dir():
    raise RuntimeError(f"Rename the existing folder {ROOT} and rerun this cell.")
actual_commit = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
if actual_commit != COMMIT:
    raise RuntimeError(f"Expected commit {COMMIT}, found {actual_commit} in {ROOT}.")
if not (ROOT / ".env").exists():
    shutil.copyfile(ROOT / ".env.example", ROOT / ".env")
for name in ("models", "datasets", "InferOutput"):
    (ROOT / name).mkdir(exist_ok=True)
print(f"Ilaria RVC {RELEASE} is ready in {ROOT}.")

### 2. Install dependencies

In [ ]:
install_marker = TOOLS / "environment-v6.ok"
runtime_env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0", "GRADIO_ANALYTICS_ENABLED": "False",
               "OMP_NUM_THREADS": "2", "MPLBACKEND": "Agg"}
if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    run("apt-get", "-q", "update")
    run("apt-get", "-qy", "install", "ffmpeg")
for binary in ("ffmpeg", "ffprobe"):
    executable = ROOT / (binary + ".exe")
    if not executable.exists():
        executable.symlink_to(shutil.which(binary))
if not all(shutil.which(binary) for binary in ("gcc", "g++", "make")):
    run("apt-get", "-q", "update")
    run("apt-get", "-qy", "install", "build-essential")

if not (PYTHON.is_file() and install_marker.is_file()):
    if not PYTHON.is_file():
        bootstrap = TOOLS / "bootstrap"
        run(sys.executable, "-m", "pip", "install", "--quiet", "--target", bootstrap, "uv==0.9.10")
        uv_env = {
            **os.environ,
            "PYTHONPATH": str(bootstrap),
            "UV_PYTHON_INSTALL_DIR": str(TOOLS / "python"),
            "UV_CACHE_DIR": str(TOOLS / "uv-cache"),
        }
        run(sys.executable, "-m", "uv", "python", "install", "3.10", env=uv_env)
        run(sys.executable, "-m", "uv", "venv", "--python", "3.10", "--seed", TOOLS / "venv", env=uv_env)

    install_log = TOOLS / "install.log"

    def pip(*args):
        with install_log.open("a", encoding="utf-8") as log:
            result = subprocess.run(
                [str(PYTHON), "-m", "pip", "--disable-pip-version-check", *args],
                stdout=log, stderr=subprocess.STDOUT,
            )
        if result.returncode:
            with install_log.open("rb") as log:
                size = log.seek(0, os.SEEK_END)
                log.seek(max(0, size - 12000))
                print(log.read().decode("utf-8", errors="replace"), flush=True)
            raise RuntimeError(f"Dependency installation failed. Full log: {install_log}")

    pip("install", "pip==24.0", "setuptools==69.5.1", "wheel==0.43.0", "numpy==1.23.5")
    print("Installing CUDA PyTorch. This can take several minutes.", flush=True)
    pip("install", "--extra-index-url", "https://download.pytorch.org/whl/cu118",
        "torch==2.0.0+cu118", "torchaudio==2.0.1+cu118")
    print("Installing audio and web interface dependencies.", flush=True)
    # Build native modules separately so compiler errors reach Kaggle's output.
    build_env = {**os.environ, "CC": shutil.which("gcc"), "CXX": shutil.which("g++"),
                 "CPLUS_INCLUDE_PATH": os.environ.get("CPLUS_INCLUDE_PATH", ""),
                 "MAX_JOBS": "2"}
    if not build_env["CPLUS_INCLUDE_PATH"]:
        build_env.pop("CPLUS_INCLUDE_PATH")
    # HuBERT uses Fairseq's CPU extensions. Keep CUDA enabled for the app.
    # Fairseq enables CUDA extensions whenever CUDA_HOME exists, even if empty.
    build_env.pop("CUDA_HOME", None)
    build_env.update(CUDA_VISIBLE_DEVICES="", FORCE_CUDA="0")
    # The PyPI sdist omits C++ files included in the upstream release.
    fairseq_source = "https://codeload.github.com/facebookresearch/fairseq/tar.gz/refs/tags/v0.12.2"
    for package, source in (("fairseq==0.12.2", fairseq_source), ("pyworld==0.3.5", "pyworld==0.3.5")):
        name, version = package.split("==", 1)
        installed = subprocess.run(
            [str(PYTHON), "-c",
             "from importlib.metadata import version; import sys; print(version(sys.argv[1]))",
             name],
            capture_output=True, text=True,
        )
        if installed.returncode == 0 and installed.stdout.strip() == version:
            print(f"Reusing {package}.", flush=True)
            continue
        cython_version = "0.29.37" if name == "fairseq" else "3.0.12"
        pip("install", f"Cython=={cython_version}")
        print(f"Building {package}...", flush=True)
        build = subprocess.run(
            [str(PYTHON), "-m", "pip", "install", "--no-build-isolation",
             "--no-deps", "--verbose", source],
            env=build_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        )
        with install_log.open("a", encoding="utf-8") as log:
            log.write(build.stdout)
        if build.returncode:
            print(build.stdout[-24000:], flush=True)
            raise RuntimeError(f"Native build failed for {package}. See the compiler output above.")
    pip("install", "--no-build-isolation",
        "fairseq==0.12.2", "hydra-core==1.0.7", "omegaconf==2.0.6",
        "gradio==3.14.0", "fastapi==0.88.0", "starlette==0.22.0", "pydantic==1.10.2",
        "uvicorn==0.20.0", "httpx==0.23.0", "python-multipart==0.0.5", "websockets==10.4",
        "altair==4.2.2", "jinja2==3.1.4", "pandas==1.5.3", "matplotlib==3.6.2", "pillow==9.5.0",
        "markdown-it-py==2.2.0", "mdit-py-plugins==0.3.3", "linkify-it-py==1.0.3",
        "numpy==1.23.5", "scipy==1.9.3", "numba==0.56.4", "llvmlite==0.39.1", "librosa==0.9.1",
        "scikit-learn==1.2.2", "resampy==0.4.3", "soundfile==0.12.1", "av==12.3.0",
        "faiss-cpu==1.7.4", "praat-parselmouth==0.4.6", "pyworld==0.3.5",
        "tensorboard==2.12.0", "protobuf==3.20.3", "onnx==1.15.0", "onnxruntime==1.16.3",
        "torchcrepe==0.0.20", "noisereduce==3.0.3", "python-dotenv==1.0.1",
        "tabulate==0.9.0", "pycryptodome==3.23.0", "psutil==6.1.1")
    pip("check")
    install_marker.touch()
else:
    print("Linux environment already installed.")

run(PYTHON, "-c", """
import torch
import torchaudio, fairseq, gradio, librosa, av, faiss, pyworld, parselmouth
from infer.modules.vc.modules import VC
from torch.utils.tensorboard import SummaryWriter
assert torch.cuda.is_available(), "PyTorch cannot detect an NVIDIA GPU"
print("GPU ready:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "| Gradio:", gradio.__version__)
""", cwd=ROOT, env={**runtime_env, "PYTHONPATH": f"{ROOT / 'ilaria'}:{ROOT}"})

### 3. Download model weights

In [ ]:
import hashlib
import time
from urllib.request import Request, urlopen

TRAINING_SAMPLE_RATES = ["32k"]  # [] for inference only, or ["32k", "40k", "48k"]
DOWNLOAD_FCPE = True
ASSET_SOURCE = "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/e6d0c1a17da07c33557852f9dfa2bd44cc75737d"
PRETRAIN_HASHES = {
    "32k": ("2332611297b8d88c7436de8f17ef5f07a2119353e962cd93cda5806d59a1133d",
            "bd7134e7793674c85474d5145d2d982e3c5d8124fc7bb6c20f710ed65808fa8a"),
    "40k": ("3b2c44035e782c4b14ddc0bede9e2f4a724d025cd073f736d4f43708453adfcb",
            "6b6ab091e70801b28e3f41f335f2fc5f3f35c75b39ae2628d419644ec2b0fa09"),
    "48k": ("b5d51f589cc3632d4eae36a315b4179397695042edc01d15312e1bddc2b764a4",
            "2269b73c7a4cf34da09aea99274dabf99b2ddb8a42cbfb065fb3c0aa9a2fc748"),
}

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def download(url, destination, expected_hash):
    if destination.is_file() and sha256_file(destination) == expected_hash:
        print(f"{destination.name}: already verified.")
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".part")
    for attempt in range(1, 4):
        print(f"Downloading {destination.name} (attempt {attempt}/3)...", flush=True)
        try:
            with urlopen(Request(url, headers={"User-Agent": "Ilaria-RVC-Kaggle/2.1"}), timeout=90) as response:
                with temporary.open("wb") as stream:
                    shutil.copyfileobj(response, stream, length=4 * 1024 * 1024)
            if sha256_file(temporary) != expected_hash:
                raise ValueError(f"SHA-256 check failed for {destination.name}.")
            temporary.replace(destination)
            print(f"{destination.name}: ready.")
            return
        except (OSError, ValueError):
            if attempt == 3:
                raise
            time.sleep(2 * attempt)
        finally:
            temporary.unlink(missing_ok=True)

weights = [
    ("hubert_base.pt", "assets/hubert/hubert_base.pt",
     "f54b40fd2802423a5643779c4861af1e9ee9c1564dc9d32f54f20b5ffba7db96"),
    ("rmvpe.pt", "assets/rmvpe/rmvpe.pt",
     "6d62215f4306e3ca278246188607209f09af3dc77ed4232efdd069798c4ec193"),
]
for rate in TRAINING_SAMPLE_RATES:
    for role, digest in zip(("G", "D"), PRETRAIN_HASHES[rate]):
        filename = f"pretrained_v2/f0{role}{rate}.pth"
        weights.append((filename, f"assets/{filename}", digest))
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=3) as executor:
    downloads = [
        executor.submit(download, f"{ASSET_SOURCE}/{filename}", ROOT / relative_destination, digest)
        for filename, relative_destination, digest in dict.fromkeys(weights)
    ]
    for task in downloads:
        task.result()

if DOWNLOAD_FCPE:
    from zipfile import ZipFile
    fcpe_target = ROOT / "assets/pitch/fcpe_runtime/torchfcpe/assets/fcpe_c_v001.pt"
    fcpe_hash = "b9aeaeb673436eeda50ceafd632aa681aa63417e52eae4207503d180c9b10015"
    if not fcpe_target.is_file() or sha256_file(fcpe_target) != fcpe_hash:
        fcpe_wheel = TOOLS / "torchfcpe-0.0.4-py3-none-any.whl"
        fcpe_url = (
            "https://files.pythonhosted.org/packages/15/98/"
            "66da447498a2835b01cb3ae851e1333035a9fbf228e5c6f3e3c4351c06b6/"
            "torchfcpe-0.0.4-py3-none-any.whl"
        )
        try:
            download(fcpe_url, fcpe_wheel, "f042c463d850d76c6f4899a0b84f0b694bb560adf05f4de951097a756d17472d")
            fcpe_target.parent.mkdir(parents=True, exist_ok=True)
            temporary = fcpe_target.with_suffix(".part")
            try:
                with ZipFile(fcpe_wheel) as archive, archive.open("torchfcpe/assets/fcpe_c_v001.pt") as model:
                    with temporary.open("wb") as stream:
                        shutil.copyfileobj(model, stream)
                if sha256_file(temporary) != fcpe_hash:
                    raise ValueError("SHA-256 check failed for FCPE.")
                temporary.replace(fcpe_target)
            finally:
                temporary.unlink(missing_ok=True)
        except (OSError, KeyError, ValueError) as error:
            print(f"FCPE unavailable ({error}). Use RMVPE.")
print("Base weights ready.")

### 4. Open Ilaria RVC

In [ ]:
import re
import signal
from html import escape
from IPython.display import HTML, Javascript, display

# PyAV accepts r/w even when the underlying stream is binary.
audio_module = ROOT / "infer/lib/audio.py"
audio_source = audio_module.read_text(encoding="utf-8")
patched_audio = audio_source.replace('av.open(i, "rb")', 'av.open(i, "r")').replace(
    'av.open(o, "wb",', 'av.open(o, "w",'
)
if patched_audio != audio_source:
    audio_module.write_text(patched_audio, encoding="utf-8")

# Decode path-based inference and training audio with Kaggle's FFmpeg.
# PyAV can inherit "rb" from the Python file object's mode.
decoder = """
def load_audio(file, sr):
    import subprocess

    path = file.strip().strip('"').strip()
    result = subprocess.run(
        ["ffmpeg", "-nostdin", "-v", "error", "-i", path,
         "-map", "0:a:0", "-ac", "1", "-ar", str(sr),
         "-c:a", "pcm_f32le", "-f", "f32le", "pipe:1"],
        capture_output=True,
    )
    if result.returncode:
        detail = result.stderr.decode("utf-8", errors="replace").strip()
        raise RuntimeError(f"Could not decode audio: {detail}")
    return np.frombuffer(result.stdout, dtype="<f4").copy()
"""
audio_source = audio_module.read_text(encoding="utf-8")
if decoder.strip() not in audio_source:
    start = audio_source.index("def load_audio(file, sr):")
    end = audio_source.find("\ndef ", start + 1)
    if end < 0:
        end = len(audio_source)
    patched_audio = audio_source[:start] + decoder.strip() + "\n\n" + audio_source[end:]
    compile(patched_audio, str(audio_module), "exec")
    audio_module.write_text(patched_audio, encoding="utf-8")

entrypoint = ROOT / "ILARIA_RVC.py"
original = "share=True, favicon_path=str(UI_ICON_PATH), prevent_thread_lock=True"
patched = (
    'share=True, favicon_path=str(UI_ICON_PATH), prevent_thread_lock=True,\n'
    '            server_name="127.0.0.1",\n'
)
source = entrypoint.read_text(encoding="utf-8")
previous_patch = patched.replace('server_name="127.0.0.1",', 'server_name="127.0.0.1", server_port=config.listen_port,')
if previous_patch in source:
    source = source.replace(previous_patch, patched, 1)
    entrypoint.write_text(source, encoding="utf-8")
if patched not in source:
    if source.count(original) != 1:
        raise RuntimeError("The launch entry point has changed.")
    entrypoint.write_text(source.replace(original, patched), encoding="utf-8")

# Limit work on the single Kaggle GPU and reduce background tunnel traffic.
source = entrypoint.read_text(encoding="utf-8")
source = source.replace(
    "app.load(refresh_training_queue, [queue_job], [queue_table, queue_status, queue_job], every=2, show_progress=False)",
    "app.load(refresh_training_queue, [queue_job], [queue_table, queue_status, queue_job], every=10, show_progress=False)",
    1,
)
cloud_launch = "if config.iscolab:\n        app.queue(concurrency_count=511, max_size=1022).launch("
source = source.replace(
    cloud_launch,
    "if config.iscolab:\n        app.queue(concurrency_count=2, max_size=32).launch(",
    1,
)
# Use the existing streaming model importer through the Gradio tunnel.
source = source.replace(
    '                if not config.iscolab:\n                    build_binary_model_upload(gr)',
    '                build_binary_model_upload(gr)',
)
start = source.index('                # Keep the existing API available to integrations using Gradio File inputs.')
end = source.index('            with gr.Accordion(', start)
source = source[:start] + source[start:end].replace('visible=config.iscolab)', 'visible=False)') + source[end:]
source = source.replace(
    '    if not config.iscolab:\n        app.load(_js=BINARY_MODEL_UPLOAD_JS)',
    '    app.load(_js=BINARY_MODEL_UPLOAD_JS)',
)
source = source.replace(
    '    if not config.iscolab:\n        register_binary_model_import(app.server_app, now_dir, update_operation_gate)',
    '    register_binary_model_import(app.server_app, now_dir, update_operation_gate)\n    if not config.iscolab:',
)
upload_module = ROOT / "ilaria/ilaria_model_upload.py"
upload_source = upload_module.read_text(encoding="utf-8")
old_origin = '        if not origin or origin.casefold() != f"{request.url.scheme}://{request.url.netloc}".casefold():'
new_origin = """        allowed_origins = {f"{request.url.scheme}://{request.url.netloc}".casefold()}
        share_url = getattr(request.app.blocks, "share_url", None)
        if share_url:
            allowed_origins.add(share_url.rstrip("/").casefold())
        if not origin or origin.casefold() not in allowed_origins:"""
if new_origin not in upload_source:
    if upload_source.count(old_origin) != 1:
        raise RuntimeError("The model upload origin check has changed.")
    upload_source = upload_source.replace(old_origin, new_origin, 1)
    compile(upload_source, str(upload_module), "exec")
    upload_module.write_text(upload_source, encoding="utf-8")

compile(source, str(entrypoint), "exec")
entrypoint.write_text(source, encoding="utf-8")

# Keep the dashboard restricted to the local UI and its own Gradio link.
dashboard = ROOT / "ilaria/ilaria_training_dashboard.py"
local_origin_test = '        if request.headers.get("origin", "").casefold() != f"{request.url.scheme}://{request.url.netloc}".casefold():'
share_origin_test = (
    '        allowed_origins = {f"{request.url.scheme}://{request.url.netloc}".casefold()}\n'
    '        share_url = getattr(request.app.blocks, "share_url", None)\n'
    '        if share_url:\n'
    '            allowed_origins.add(share_url.rstrip("/").casefold())\n'
    '        if request.headers.get("origin", "").casefold() not in allowed_origins:'
)
source = dashboard.read_text(encoding="utf-8")
if source.count(share_origin_test) != 2:
    if source.count(local_origin_test) != 2:
        raise RuntimeError("The Training Dashboard access check has changed.")
    dashboard.write_text(source.replace(local_origin_test, share_origin_test), encoding="utf-8")

log_file = TOOLS / "startup.log"

def log_tail(lines=30):
    if not log_file.is_file():
        return ""
    with log_file.open("rb") as stream:
        size = stream.seek(0, os.SEEK_END)
        stream.seek(max(0, size - 65536))
        return "\n".join(stream.read().decode("utf-8", errors="replace").splitlines()[-lines:])

def stop_ui(process):
    if process is not None and process.poll() is None:
        for stop_signal, timeout in ((signal.SIGINT, 15), (signal.SIGTERM, 10), (signal.SIGKILL, 5)):
            try:
                os.killpg(process.pid, stop_signal)
            except ProcessLookupError:
                return
            try:
                process.wait(timeout=timeout)
                return
            except subprocess.TimeoutExpired:
                continue
        raise RuntimeError("The process is still running. Use Stop Session in Kaggle.")

ui_process = globals().get("ui_process")
if ui_process is not None and ui_process.poll() is None:
    print("Restarting Ilaria RVC to load the current audio decoder.", flush=True)
    stop_ui(ui_process)

# Test the patched decoder in a fresh Python process before opening Gradio.
run(PYTHON, "-c", """
import tempfile
import wave
from pathlib import Path
from infer.lib.audio import load_audio

with tempfile.TemporaryDirectory() as directory:
    sample = Path(directory) / "audio check.wav"
    with wave.open(str(sample), "wb") as audio:
        audio.setparams((1, 2, 8000, 800, "NONE", "not compressed"))
        audio.writeframes(bytes(1600))
    decoded = load_audio(str(sample), 16000)
    assert decoded.shape == (1600,), decoded.shape
print("FFmpeg audio decoder verified.", flush=True)
""", cwd=ROOT, env=runtime_env)

if ui_process is None or ui_process.poll() is not None:
    launch_env = {**runtime_env, "PYTHONUNBUFFERED": "1"}
    with log_file.open("w", encoding="utf-8") as log:
        ui_process = subprocess.Popen(
            [str(PYTHON), "-u", str(entrypoint), "--colab", "--noautoopen"],
            cwd=ROOT, env=launch_env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
        )

print("Starting Ilaria RVC and opening the Gradio link...", flush=True)
public_url = None
deadline = time.monotonic() + 150
while time.monotonic() < deadline:
    if ui_process.poll() is not None:
        raise RuntimeError(f"Ilaria RVC closed during startup. Last log lines:\n{log_tail()}")
    tail = log_tail(100)
    match = re.search(r"Running on public URL:\s*(https://[a-zA-Z0-9-]+\.(?:gradio\.live|gradio\.app))\b", tail)
    if match:
        public_url = match[1]
        break
    if "Could not create share link" in tail:
        break
    time.sleep(1)
if public_url is None:
    stop_ui(ui_process)
    raise RuntimeError(
        "The Gradio link is unavailable. Enable Internet and rerun this cell.\n"
        + log_tail()
    )
safe_url = escape(public_url, quote=True)
display(HTML(
    f'<a href="{safe_url}" target="_blank" rel="noopener noreferrer" '
    'style="display:inline-block;padding:10px 18px;border-radius:8px;'
    'background:#ffc0cb;color:#3f2340;font-weight:700;text-decoration:none">'
    'Open Ilaria RVC</a>'
    '<p>Keep this Kaggle session running while you use the app.</p>'
))
display(Javascript("window.open(" + repr(public_url) + ", '_blank', 'noopener,noreferrer');"))


### Export results · optional

Set `EXPORT = True`. To include resumable checkpoints, click **Stop and save** in Ilaria RVC first and fill in `TRAINING_MODEL`.

In [ ]:
EXPORT = False
TRAINING_MODEL = ""  # Leave blank to exclude training files.

if EXPORT:
    from zipfile import ZipFile, ZIP_STORED
    from IPython.display import FileLink

    folders = [ROOT / "models", ROOT / "InferOutput"]
    if TRAINING_MODEL:
        name = TRAINING_MODEL.strip()
        if not name or Path(name).name != name or name in {".", "..", "mute"} or "\\" in name:
            raise ValueError("Enter a model name instead of a path.")
        experiment = ROOT / "logs" / name
        if not experiment.is_dir():
            raise FileNotFoundError(f"Training files not found: {name}")
        folders.append(experiment)

    files = [
        file for folder in folders for file in folder.rglob("*")
        if file.is_file() and not file.is_symlink() and file.resolve().is_relative_to(ROOT)
        and (folder.name != "models" or file.suffix.lower() in {".pth", ".index"})
    ]
    if not files:
        print("There are no results to export.")
    else:
        archive = WORK / "Ilaria-RVC-results.zip"
        partial = archive.with_suffix(".part")
        try:
            with ZipFile(partial, "w", compression=ZIP_STORED) as zip_file:
                for file in sorted(files):
                    zip_file.write(file, file.relative_to(ROOT))
            partial.replace(archive)
        finally:
            partial.unlink(missing_ok=True)
        link_path = os.path.relpath(archive, start=Path.cwd())
        display(FileLink(link_path))
        print(f"Also available in Output > {archive.name}")

### Stop or restart · optional

Save any running training first. Set `STOP = True` to stop the UI. Run step 4 again to restart.

In [ ]:
STOP = False
if STOP:
    stop_ui(globals().get("ui_process"))
    print("Ilaria RVC stopped. Run section 4 to reopen it.")